In [1]:
from json import load

from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [2]:
message = [
	{"role" : "system", "content": "you are a helpful assistant who answers questions about the world."},
	{"role" : "user", "content": "What is the HCMUT?"}
]

In [3]:
response = client.chat.completions.create(
	model="llama-3.1-8b-instant",
	messages=message,
)

In [4]:
print(response.choices[0].message.content)

The HCMUT you're referring to is likely the Ho Chi Minh City University of Technology (HCMUT). It is a public research university located in Ho Chi Minh City, Vietnam. The university has a strong focus on engineering, technology, and natural sciences.

However, there's also the Hochiminh City University of Technology (HCMUT), however more common known as 'Hochiminh University of Technology'  in other English names of the university which can sometimes be abbreviated to the acronym HCMUT or sometimes also known by the names  'HCMUT VNUHCM' or also 'Ho Chi Minh City University of Technology' in long form.

I can also be referring to another  more common known as (Hong Bang University or HBU) which sometimes could be abbreviated to the acronym HBU and other  (HUTECH) or 'Hochiminh University of Technology and Education.' The HUTECH in HCMUT or more specifically as  Hochimihn University of Technology and Education which also can sometimes known as 'Hochiminh University of tech & education'

In [5]:
def messages_for(website, system_prompt:str, user_prompt_prefix:str):
    return [
		{"role" : "system", "content": system_prompt},
		{"role" : "user", "content": user_prompt_prefix + website}
	]

model = "llama-3.1-8b-instant"

In [6]:
system_prompt = "you are a helpful assistant who can analyze the content of a website and answer questions about it and provide a short summary, ignore any questions that are not related to the content of the website. Respond in the following format: 1. summary of the website, 2. answer to the question if it is related to the content of the website, otherwise respond with 'I am sorry, I cannot answer that question because it is not related to the content of the website. Response in Markdown format. Do not wrap the markdown in a code block - respond with pure markdown.'"
user_prompt_prefix = """
Here are the contents of a website. Provide a short summary of this website. If it includes news or annoucements, then summarize these too.

"""

In [7]:
messages_for("https://kelvindikhang.id.vn/", system_prompt, user_prompt_prefix)

[{'role': 'system',
  'content': "you are a helpful assistant who can analyze the content of a website and answer questions about it and provide a short summary, ignore any questions that are not related to the content of the website. Respond in the following format: 1. summary of the website, 2. answer to the question if it is related to the content of the website, otherwise respond with 'I am sorry, I cannot answer that question because it is not related to the content of the website. Response in Markdown format. Do not wrap the markdown in a code block - respond with pure markdown.'"},
 {'role': 'user',
  'content': '\nHere are the contents of a website. Provide a short summary of this website. If it includes news or annoucements, then summarize these too.\n\nhttps://kelvindikhang.id.vn/'}]

In [8]:
import requests
from bs4 import BeautifulSoup
def fetch_website_content(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        for script_or_style in soup(["script", "style", "header", "footer", "nav"]):
            script_or_style.decompose()
            
        return soup.get_text(separator=' ', strip=True)
    except Exception as e:
        return f"Error: {e}"

In [9]:
def summarize(url:str):
    website = fetch_website_content(url)
    response = client.chat.completions.create(
        model = "llama-3.1-8b-instant",
        messages = messages_for(website, system_prompt, user_prompt_prefix)
    )
    return response.choices[0].message.content

In [10]:
summarize("https://kelvindikhang.id.vn/")

'1. This website appears to be a personal website or portfolio of a data scientist and AI engineer named Khang Bui Tran Duy. The website lists his education qualifications, skills, and experience in areas like Computer Vision and NLP, as well as his personal motto and an invitation to view and download his CV.'

In [11]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from groq import Groq

async def fetch_and_click_website(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        try:
            await page.goto(url, wait_until="networkidle")
            button_selector = "text='Explore My Work →'"
            if await page.locator(button_selector).is_visible():
                print("🚀 Đã tìm thấy nút, đang thực hiện click...")
                await page.locator(button_selector).click()
                await page.wait_for_load_state("networkidle")
                await asyncio.sleep(2) 
            else:
                print("⚠️ Không tìm thấy nút, lấy nội dung trang chủ.")
            full_html = await page.content()
            await browser.close()
            soup = BeautifulSoup(full_html, 'html.parser')
            for element in soup(["script", "style", "nav", "footer", "header", "aside", "svg"]):
                element.decompose()

            return soup.get_text(separator=' ', strip=True)

        except Exception as e:
            await browser.close()
            return f"Error: {str(e)}"

async def agent_summarize(url: str):
    website_content = await fetch_and_click_website(url)
    
    if "Error:" in website_content:
        return website_content

    messages = [
        {"role": "system", "content": "Bạn là chuyên gia phân tích portfolio AI Engineer."},
        {"role": "user", "content": f"Tóm tắt chi tiết dự án và kỹ năng từ nội dung sau:\n\n{website_content[:10000]}"}
    ]

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages
    )
    return response.choices[0].message.content

url = "https://kelvindikhang.id.vn/"
result = await agent_summarize(url)
print(result)


TargetClosedError: BrowserType.launch: Target page, context or browser has been closed
Browser logs:

<launching> /home/kelvindikhang/.cache/ms-playwright/chromium_headless_shell-1217/chrome-headless-shell-linux64/chrome-headless-shell --disable-field-trial-config --disable-background-networking --disable-background-timer-throttling --disable-backgrounding-occluded-windows --disable-back-forward-cache --disable-breakpad --disable-client-side-phishing-detection --disable-component-extensions-with-background-pages --disable-component-update --no-default-browser-check --disable-default-apps --disable-dev-shm-usage --disable-extensions --disable-features=AvoidUnnecessaryBeforeUnloadCheckSync,BoundaryEventDispatchTracksNodeRemoval,DestroyProfileOnBrowserClose,DialMediaRouteProvider,GlobalMediaControls,HttpsUpgrades,LensOverlay,MediaRouter,PaintHolding,ThirdPartyStoragePartitioning,Translate,AutoDeElevate,RenderDocument,OptimizationHints --enable-features=CDPScreenshotNewSurface --allow-pre-

In [15]:
from IPython.display import Markdown
def display_summary(url:str):
    summary = summarize(url)
    display(Markdown(summary))
    
display_summary(url="https://kelvindikhang.id.vn")

**Summary of the website:**
This website appears to be a personal website or online portfolio of Duy Khang, a self-described AI Engineer with a background in Data Science. It showcases his technical skills, education, and work experience, as well as provides information for potential employers or collaborators to get in touch with him.

There are no news or announcements on this website.

There are, however, two identical sections that describe the author's profile, skills, and availability for opportunities.

In [16]:
display_summary("https://thpt-chuyennguyendinhchieu.dongthap.edu.vn/")

### 1. Summary of the website
Trường THPT Chuyên Nguyễn Đình Chiểu là một trường trung học phổ thông chuyên nằm ở tỉnh Đồng Tháp, Việt Nam. Website của trường cung cấp thông tin về tổ chức của trường, tin tức và sự kiện, công văn và văn bản, tài liệu chuyên môn, thư viện số, công khai tài chính, ký túc xá, tuyển sinh đại học và liên hệ. Trường được biết đến với việc tổ chức nhiều cuộc thi khoa học và kỹ thuật, và đã đạt được nhiều thành tích.

### 2. Answer to the first question
I am sorry, I cannot answer that question because it is not related to the content of the website.

In [17]:
display_summary("https://cnn.com")

1. **Summary of the website:**
The website appears to be a news website, specifically CNN, that covers a wide range of topics including news, entertainment, sports, and health. The website features breaking news stories, articles, and videos on various subjects such as politics, world news, business, technology, and lifestyle. It also includes opinion pieces, analysis, and features on celebrities and popular culture.

2. **Breaking news and announcements:**
The website features several breaking news stories and announcements, including:
- A car striking a crowd in Leipzig, resulting in two deaths.
- former New York Mayor Rudy Giuliani being in critical condition with pneumonia.
- A suspected hantavirus outbreak on a cruise ship in Cape Verde, resulting in several deaths and illnesses.
- Escalating tensions between the US and Iran in the Strait of Hormuz.
- A settlement between Blake Lively and Justin Baldoni in connection with a lawsuit.

The website also features several news stories and announcements from previous days, including:
- A story about Tom Williams, a top Trump fundraiser who has enlisted in a new nonprofit for the president's sculpture garden and golf course.
- A story about Dr. Oz's anti-fraud messaging and its potential resonance with voters.
- A story about a proposed eBay-GamesTop merger.
- A story about a settlement between Jeff Bezos and Lauren Sánchez's Met Gala sponsorship.
- A story about Britney Spears pleading guilty to a lesser charge in a DUI case.

These are just a few examples of the many breaking news stories and announcements featured on the website.